# FRP-XGBoost and PLM-FRP — matched reconstructions

Reconstructions of **both** published ferroptosis predictors, on **identical sequences**:

| Model | Features (full published set) | Selection | Classifier |
|---|---|---|---|
| **FRP-XGBoost** (Lin et al.) | AAC (20) + CKSAAP gap 0–4 (2000) + codon-DDE (400) + GTPC (125) = 2545 | IFS (gain-ranked) | XGBoost |
| **PLM-FRP** (Zhou & Wang) | codon-DDE (400) + mean-pooled ESM-2-650M (1280) = 1680 | IFS (gain-ranked) | XGBoost |

Both draw from the **same training pool and external cohort** (ESM3∩ESM2 intersection, housekeeping
holdout) and share the **same internal-test split**, so the comparison isolates each method's own
feature construction. **Incremental Feature Selection (IFS):** rank features by XGBoost gain, add them
in blocks, keep the count that maximizes cross-validated AUROC.

**Run:** Runtime → **GPU**, Run all (ESM-2 embedding + IFS ≈ 30–60 min). Outputs → `MyDrive/JR_Ferro/baselines/`.


In [ ]:
!pip install -q xgboost transformers biopython openpyxl

In [ ]:
import os, re, gzip, io, time, gc
import numpy as np, pandas as pd, torch
from pathlib import Path
from Bio import SeqIO
import xgboost as xgb
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, roc_auc_score,
                             average_precision_score, confusion_matrix)
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/JR_Ferro')
ESM2_DIR     = PROJECT_ROOT / 'ESM_Embedding'      # precomputed ESM2 features + metadata
EMBED_DIR    = PROJECT_ROOT / 'ESM3_Embedding'     # ESM3 metadata (for the ESM3∩ESM2 intersection)
EXTERNAL_DIR = PROJECT_ROOT / 'Data/Ferro/external'
NEG_DIR      = EXTERNAL_DIR / 'negative'
POS_FASTA    = PROJECT_ROOT / 'Data/Ferro/ferro_pos_rep_seq.fasta'
NEG_FASTA    = PROJECT_ROOT / 'Data/Ferro/ferro_neg_rep_seq.fasta'
OUT          = PROJECT_ROOT / 'baselines'; OUT.mkdir(parents=True, exist_ok=True)

random_seed = 42; np.random.seed(random_seed)
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
XGB_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_SEQ_LEN, PER_GENE_CAP, HOLDOUT_FRAC = 1536, 1500, 0.30
KMAX      = 4                    # CKSAAP gaps 0..KMAX  -> 400*(KMAX+1) features
IFS_MAX, IFS_STEP, IFS_SUB, IFS_FOLDS = 500, 10, 25000, 3   # incremental feature selection
EXCLUDE_NEG = {'NLRP3','FTL','YY1AP1'}; EXCLUDE_POS = {'NLRP3'}
VALID_AA    = set('ACDEFGHIKLMNPQRSTVWY')
# Housekeeping genes excluded from the cohort
HK_HOLDOUT  = {'SKP1','FEN1','RPL21','TBP','GAPDH','PPIA','RPS2','EXO1','DES',
               'RDX','FLNA','ALDOA','PGK1','HSPA8','MAPT'}
PATHWAY = {
    'ANXA5':'Apoptosis','APAF1':'Apoptosis','BBC3':'Apoptosis','BCL2':'Apoptosis',
    'CASP3':'Apoptosis','CASP7':'Apoptosis','DFFB':'Apoptosis','FADD':'Apoptosis',
    'CASP8':'Necroptosis','MLKL':'Necroptosis','RIPK1':'Necroptosis','RIPK3':'Necroptosis',
    'TNFRSF1A':'Necroptosis','TRADD':'Necroptosis','ZBP1':'Necroptosis',
    'AIM2':'Pyroptosis','CASP1':'Pyroptosis','CASP4':'Pyroptosis','CASP5':'Pyroptosis',
    'GSDMD':'Pyroptosis','GSDME':'Pyroptosis','IL1B':'Pyroptosis',
}
for p in [ESM2_DIR, EMBED_DIR, EXTERNAL_DIR, NEG_DIR, POS_FASTA, NEG_FASTA]:
    assert Path(p).exists(), f'Missing: {p}'
print('Paths OK.  device:', device, '| XGB:', XGB_DEVICE)

In [ ]:
# Parsers + hard-neg gene-disjoint split + external cohort
def gene_from_fname(name):
    s = re.sub(r'\.xlsx$', '', name); s = re.sub(r'\.xlsx$', '', s)
    s = re.sub(r'^uniparc_', '', s)
    return re.split(r'_AND_|_20\d\d', s)[0].strip('_')

def read_xlsx_any(path):
    raw = Path(path).read_bytes()
    if raw[:2] == b'\x1f\x8b': raw = gzip.decompress(raw)
    return pd.read_excel(io.BytesIO(raw), engine='openpyxl')

def parse_xlsx(path, label, pathway):
    df = read_xlsx_any(path); df.columns = [c.strip() for c in df.columns]
    seq_col = next((c for c in df.columns if 'seq'   in c.lower()), df.columns[1])
    gene = gene_from_fname(path.name); rows = []
    for _, r in df.iterrows():
        seq = ''.join(c for c in str(r[seq_col]).strip().upper() if c in VALID_AA)
        if len(seq) >= 10:
            rows.append({'gene':gene,'pathway':pathway,'sequence':seq,'label':label})
    return rows

records = []
for path in sorted(NEG_DIR.glob('*.xlsx')):
    g = gene_from_fname(path.name)
    if g in EXCLUDE_NEG: continue
    records.extend(parse_xlsx(path, label=0, pathway=PATHWAY.get(g, 'Other')))
rcd = pd.DataFrame(records)
rng = np.random.default_rng(random_seed)
def cap(group):
    if len(group) <= PER_GENE_CAP: return group
    return group.iloc[rng.permutation(len(group))[:PER_GENE_CAP]]
rcd = rcd.groupby('gene', group_keys=False).apply(cap).reset_index(drop=True)

gene_pw = rcd[['gene','pathway']].drop_duplicates().sort_values('gene').reset_index(drop=True)
holdout_genes, train_genes = [], []
for pw, sub in gene_pw.groupby('pathway'):
    genes = sub['gene'].to_numpy()
    order = np.random.default_rng(random_seed).permutation(len(genes))
    n_hold = max(1, round(len(genes) * HOLDOUT_FRAC))
    holdout_genes += list(genes[order[:n_hold]]); train_genes += list(genes[order[n_hold:]])
train_genes, holdout_genes = set(train_genes), set(holdout_genes)
rcd_train = rcd[rcd['gene'].isin(train_genes)].reset_index(drop=True)
rcd_hold  = rcd[rcd['gene'].isin(holdout_genes)].reset_index(drop=True)

ext_records = []
for path in sorted(EXTERNAL_DIR.glob('*.xlsx')):
    if gene_from_fname(path.name) in EXCLUDE_POS: continue
    ext_records.extend(parse_xlsx(path, label=1, pathway='ferroptosis'))
extval = pd.concat([pd.DataFrame(ext_records), rcd_hold], ignore_index=True)
extval = extval.groupby('gene', group_keys=False).apply(cap).reset_index(drop=True)
print(f'Hard-neg train genes: {len(train_genes)} | held-out death genes: {len(holdout_genes)}')
print(f'External set: {len(extval)} seqs '
      f'({(extval.label==1).sum()} pos / {(extval.label==0).sum()} neg), {extval.gene.nunique()} genes')

In [ ]:
# Descriptors (iFeature-standard), ESM-2 embedding, IFS, XGBoost, metrics
AA = "ACDEFGHIKLMNPQRSTVWY"; AIDX = {a:i for i,a in enumerate(AA)}
CODON = dict(zip(AA, [4,2,2,2,2,4,2,3,2,6,1,2,4,2,6,6,4,4,1,2]))
# GTPC: 20 AA -> 5 physicochemical groups (iFeature scheme)
GRP = {'G':0,'A':0,'V':0,'L':0,'M':0,'I':0,'F':1,'Y':1,'W':1,'K':2,'R':2,'H':2,
       'D':3,'E':3,'S':4,'T':4,'C':4,'P':4,'N':4,'Q':4}
GIDX = np.array([GRP[a] for a in AA])
_C = np.array([CODON[a] for a in AA]) / 61.0; _Tm = np.outer(_C, _C).ravel()   # DDE expectation

def _idx(s):
    s = ''.join(c for c in str(s).upper() if c in AIDX)
    return np.fromiter((AIDX[c] for c in s), np.int64, len(s))

def dde_feats(seqs):                                         # PLM-FRP DDE, 400-D
    X = np.zeros((len(seqs), 400), np.float32)
    for k, s in enumerate(seqs):
        idx = _idx(s); L = len(idx)
        if L < 2: continue
        di  = idx[:-1] * 20 + idx[1:]
        dpc = np.bincount(di, minlength=400).astype(np.float32) / (L - 1)
        Tv  = _Tm * (1 - _Tm) / (L - 1)
        X[k] = (dpc - _Tm) / np.sqrt(Tv + 1e-12)
    return X

def frp_feats(seqs, kmax=KMAX):                             # FRP-XGBoost: AAC+CKSAAP+DDE+GTPC
    D = 20 + 400*(kmax+1) + 400 + 125
    X = np.zeros((len(seqs), D), np.float32)
    for k, s in enumerate(seqs):
        idx = _idx(s); L = len(idx); p = 0
        if L < 1: continue
        X[k, p:p+20] = np.bincount(idx, minlength=20) / L; p += 20            # AAC
        for g in range(kmax+1):                                              # CKSAAP gap g
            if L-g-1 > 0:
                di = idx[:L-g-1]*20 + idx[g+1:]
                X[k, p:p+400] = np.bincount(di, minlength=400) / (L-g-1)
            p += 400
        if L >= 2:                                                           # DDE
            di = idx[:-1]*20 + idx[1:]
            dpc = np.bincount(di, minlength=400).astype(np.float32) / (L-1)
            Tv  = _Tm*(1-_Tm)/(L-1)
            X[k, p:p+400] = (dpc - _Tm) / np.sqrt(Tv + 1e-12)
        p += 400
        if L >= 3:                                                           # GTPC
            gg = GIDX[idx]; tri = gg[:-2]*25 + gg[1:-1]*5 + gg[2:]
            X[k, p:p+125] = np.bincount(tri, minlength=125).astype(np.float32) / (L-2)
    return X

def esm2_embed(seqs, model, tok, B=8):
    V = []
    with torch.no_grad():
        for i in range(0, len(seqs), B):
            ch = [s[:1022] for s in seqs[i:i+B]]
            enc = tok(ch, return_tensors='pt', padding=True, truncation=True, max_length=1024).to(device)
            with torch.amp.autocast(device.type, enabled=(device.type=='cuda')):
                out = model(**enc).last_hidden_state
            m = enc['attention_mask'].clone().unsqueeze(-1).float(); m[:, 0] = 0    # drop CLS
            V.append(((out.float() * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy())
            if i % 1600 == 0: print(f'  ESM2 {i}/{len(seqs)}', flush=True)
    return np.concatenate(V).astype(np.float32)

def xgb_fit(X, y, n_est=400):
    m = xgb.XGBClassifier(n_estimators=n_est, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, tree_method='hist', device=XGB_DEVICE,
        eval_metric='logloss', n_jobs=-1, random_state=random_seed)
    m.fit(X, y); return m

def ifs_select(X, y, tag=''):
    """Rank features by XGBoost gain, then add in blocks of IFS_STEP; keep the count that
    maximizes CV AUROC (CV on a class-balanced subsample for speed)."""
    order = np.argsort(xgb_fit(X, y).feature_importances_)[::-1]
    mx = min(IFS_MAX, X.shape[1])
    if IFS_SUB and len(y) > IFS_SUB:
        r = np.random.default_rng(random_seed)
        idx = np.concatenate([r.choice(np.where(y==c)[0], min(int((y==c).sum()), IFS_SUB//2),
                                       replace=False) for c in (0,1)])
        Xc, yc = X[idx], y[idx]
    else:
        Xc, yc = X, y
    skf = StratifiedKFold(IFS_FOLDS, shuffle=True, random_state=random_seed)
    curve = []
    for n in range(IFS_STEP, mx+1, IFS_STEP):
        s = order[:n]; a = []
        for tr, va in skf.split(Xc[:, s], yc):
            mm = xgb_fit(Xc[tr][:, s], yc[tr], n_est=200)
            a.append(roc_auc_score(yc[va], mm.predict_proba(Xc[va][:, s])[:, 1]))
        curve.append((n, float(np.mean(a))))
        print(f'  [{tag}] IFS n={n:4d}  CV AUROC {np.mean(a):.4f}', flush=True)
    best_n = max(curve, key=lambda t: t[1])[0]
    print(f'  [{tag}] -> selected {best_n} features (best CV AUROC {max(c[1] for c in curve):.4f})')
    return order[:best_n], best_n, pd.DataFrame(curve, columns=['n_features','cv_auroc'])

def full_metrics(model, X, y):
    p = model.predict_proba(X)[:, 1]; pred = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    m = {'accuracy': accuracy_score(y, pred), 'auroc': roc_auc_score(y, p),
         'ap': average_precision_score(y, p),
         'sensitivity': tp/(tp+fn) if (tp+fn) else float('nan'),
         'specificity': tn/(tn+fp) if (tn+fp) else float('nan'),
         'n': int(len(y)), 'n_pos': int(y.sum())}
    return m, p, pred

In [ ]:
# Training pool: intersection (pos+HK) + hard-neg — one basis for both models
meta   = pd.read_csv(ESM2_DIR / 'sequence_metadata.csv')
X_esm2 = np.load(ESM2_DIR / 'esm_features.npy').astype(np.float32)
y_all  = np.load(ESM2_DIR / 'labels.npy').astype(np.int32)
assert len(meta) == len(X_esm2) == len(y_all), 'metadata / feature length mismatch'

h2seq = {}
for fa in [POS_FASTA, NEG_FASTA]:
    for rec in SeqIO.parse(str(fa), 'fasta'): h2seq[rec.description] = str(rec.seq)
seqs = [h2seq.get(h, '') for h in meta['header']]
meta['gene_clean'] = meta['gene'].astype(str).str.split('_AND_').str[0]

esm3_headers = set(pd.read_csv(EMBED_DIR / 'sequence_metadata.csv')['header'])
in_esm3 = meta['header'].isin(esm3_headers)
print(f'ESM3∩ESM2 intersection: {int(in_esm3.sum())} / {len(meta)} rows')

ext_genes = set(extval['gene'])
keep = ((~meta['gene_clean'].isin(HK_HOLDOUT))
        & (~meta['gene_clean'].isin(ext_genes))
        & in_esm3).to_numpy()
sel = np.where(keep)[0]
def cln(s): return ''.join(c for c in str(s).upper() if c in VALID_AA)[:MAX_SEQ_LEN]

# keep only rows with a matched, valid sequence -> IDENTICAL basis for BOTH models
orig_seq, orig_y, orig_gene, orig_rows = [], [], [], []
gc_gene = meta['gene_clean'].to_numpy()
for i in sel:
    s = cln(seqs[i])
    if len(s) >= 10:
        orig_seq.append(s); orig_y.append(int(y_all[i]))
        orig_gene.append(gc_gene[i]); orig_rows.append(i)
orig_esm2 = X_esm2[np.array(orig_rows)]; del X_esm2; gc.collect()

hn_seq = [cln(s) for s in rcd_train['sequence']]
hn_g   = rcd_train['gene'].to_numpy()
hk     = [j for j, s in enumerate(hn_seq) if len(s) >= 10]
hn_seq = [hn_seq[j] for j in hk]; hn_gene = hn_g[hk]; hn_y = np.zeros(len(hn_seq), int)

ext_seq, ext_y, ext_gene = [], [], []
for s, l, g in zip(extval['sequence'], extval['label'], extval['gene']):
    s2 = cln(s)
    if len(s2) >= 10: ext_seq.append(s2); ext_y.append(int(l)); ext_gene.append(g)
ext_y = np.array(ext_y); ext_gene = np.array(ext_gene)

train_seq  = orig_seq + hn_seq
train_y    = np.concatenate([np.array(orig_y), hn_y])
train_gene = np.concatenate([np.array(orig_gene), hn_gene])
# ONE internal-test split, shared by both models
itr, ite = train_test_split(np.arange(len(train_y)), test_size=0.20,
                            stratify=train_y, random_state=random_seed)
print(f'Train: {len(train_seq)} seqs ({train_y.mean():.3f} pos)  '
      f'[{len(orig_seq)} intersection + {len(hn_seq)} hard-neg]   External: {len(ext_seq)} ({ext_y.mean():.3f} pos)')
print(f'Internal-test split: {len(itr)} train / {len(ite)} test (shared by both models)')

## FRP-XGBoost — AAC + CKSAAP + DDE + GTPC → IFS → XGBoost

Feature selection is run once on the full training pool; the resulting feature set is used for
both external (trained on the full pool — gene-disjoint, clean) and the internal-test split.
The internal number is a within-distribution reference and is homology-influenced; **external is
the fair test.**

In [ ]:
# FRP-XGBoost: full descriptor set -> IFS -> XGBoost -> evaluate
t0 = time.time()
Xtr_frp = frp_feats(train_seq); Xex_frp = frp_feats(ext_seq)
print(f'FRP features: train {Xtr_frp.shape}, external {Xex_frp.shape}  ({time.time()-t0:.0f}s)')

sel_frp, nsel_frp, curve_frp = ifs_select(Xtr_frp, train_y, tag='FRP')
curve_frp.to_csv(OUT / 'frp_ifs_curve.csv', index=False)

# external (primary): train on full pool with selected features
clf_frp = xgb_fit(Xtr_frp[:, sel_frp], train_y)
m_frp_ext, p_frp_e, pred_frp_e = full_metrics(clf_frp, Xex_frp[:, sel_frp], ext_y)
# internal test: retrain on the 80% slice
m_frp_rand, p_frp_r, pred_frp_r = full_metrics(
    xgb_fit(Xtr_frp[itr][:, sel_frp], train_y[itr]), Xtr_frp[ite][:, sel_frp], train_y[ite])
print(f'FRP-XGBoost  ({nsel_frp} feats)  test AUROC {m_frp_rand["auroc"]:.4f}  |  '
      f'external AUROC {m_frp_ext["auroc"]:.4f} sens {m_frp_ext["sensitivity"]:.3f} spec {m_frp_ext["specificity"]:.3f}')
del Xtr_frp, Xex_frp; gc.collect()

## PLM-FRP — codon-DDE + mean-pooled ESM-2 → IFS → XGBoost

The same sequences and the same internal-test split. The intersection rows reuse the precomputed
ESM-2 features; hard-neg and external are embedded live (a precomputed-vs-live cosine check guards
against a pooling mismatch).

In [ ]:
# PLM-FRP: ESM-2 (precomputed intersection + live hard-neg/external) + DDE
tok2 = AutoTokenizer.from_pretrained("facebook/esm2_t33_650M_UR50D")
esm2 = AutoModel.from_pretrained("facebook/esm2_t33_650M_UR50D").to(device).eval()

_chk = list(range(min(200, len(orig_seq))))
_live = esm2_embed([orig_seq[i] for i in _chk], esm2, tok2); _pre = orig_esm2[_chk]
_cos = ((_live*_pre).sum(1) / (np.linalg.norm(_live,axis=1)*np.linalg.norm(_pre,axis=1)+1e-9)).mean()
print(f'Precomputed-vs-live ESM2 cosine (n={len(_chk)}): {_cos:.4f}  '
      + ('OK' if _cos > 0.98 else 'MISMATCH: re-embed the pool live for a clean comparison'), flush=True)

t0 = time.time()
esm2_hn = esm2_embed(hn_seq, esm2, tok2)
esm2_ex = esm2_embed(ext_seq, esm2, tok2)
del esm2; gc.collect()
if device.type == 'cuda': torch.cuda.empty_cache()
print(f'Live ESM-2 embedding done ({(time.time()-t0)/60:.1f} min)')

train_esm2 = np.vstack([orig_esm2, esm2_hn]).astype(np.float32)
Xtr_plm = np.hstack([train_esm2,        dde_feats(train_seq)]).astype(np.float32)   # [ESM2 | DDE]
Xex_plm = np.hstack([esm2_ex,           dde_feats(ext_seq)]).astype(np.float32)
print(f'PLM features: train {Xtr_plm.shape}, external {Xex_plm.shape}')

sel_plm, nsel_plm, curve_plm = ifs_select(Xtr_plm, train_y, tag='PLM')
curve_plm.to_csv(OUT / 'plmfrp_ifs_curve.csv', index=False)

clf_plm = xgb_fit(Xtr_plm[:, sel_plm], train_y)
m_plm_ext, p_plm_e, pred_plm_e = full_metrics(clf_plm, Xex_plm[:, sel_plm], ext_y)
m_plm_rand, p_plm_r, pred_plm_r = full_metrics(
    xgb_fit(Xtr_plm[itr][:, sel_plm], train_y[itr]), Xtr_plm[ite][:, sel_plm], train_y[ite])
print(f'PLM-FRP  ({nsel_plm} feats)  test AUROC {m_plm_rand["auroc"]:.4f}  |  '
      f'external AUROC {m_plm_ext["auroc"]:.4f} sens {m_plm_ext["sensitivity"]:.3f} spec {m_plm_ext["specificity"]:.3f}')
del Xtr_plm, Xex_plm; gc.collect()

In [ ]:
# Save combined metrics + per-seq + external per-gene (both models)
res = pd.DataFrame([
    {'model':'FRP-XGBoost','n_features':nsel_frp,'split':'random',   **m_frp_rand},
    {'model':'FRP-XGBoost','n_features':nsel_frp,'split':'external', **m_frp_ext},
    {'model':'PLM-FRP',    'n_features':nsel_plm,'split':'random',   **m_plm_rand},
    {'model':'PLM-FRP',    'n_features':nsel_plm,'split':'external', **m_plm_ext},
])
res = res[['model','n_features','split','accuracy','auroc','ap',
           'sensitivity','specificity','n','n_pos']].round(4)
res.to_csv(OUT / 'baselines_metrics.csv', index=False)

# per-sequence predictions
gtr_ite = train_gene[ite]
pd.DataFrame({'gene':gtr_ite,'y_true':train_y[ite],'frp_prob':p_frp_r,'plm_prob':p_plm_r}
             ).to_csv(OUT / 'baselines_random_predictions.csv', index=False)
pd.DataFrame({'gene':ext_gene,'y_true':ext_y,'frp_prob':p_frp_e,'plm_prob':p_plm_e}
             ).to_csv(OUT / 'baselines_external_predictions.csv', index=False)

# per-gene on external (both models)
def pergene(prob, pred):
    d = pd.DataFrame({'gene':ext_gene,'y_true':ext_y,'y_pred':pred,'y_prob':prob})
    g = (d.groupby('gene').apply(lambda x: pd.Series({
            'label':int(x['y_true'].iloc[0]),'n':len(x),
            'accuracy':(x['y_pred']==x['y_true']).mean(),'mean_prob':x['y_prob'].mean()}))
         .reset_index())
    g['metric'] = np.where(g['label']==1,'sensitivity','specificity')
    return g
pg_frp = pergene(p_frp_e, pred_frp_e); pg_frp['model']='FRP-XGBoost'
pg_plm = pergene(p_plm_e, pred_plm_e); pg_plm['model']='PLM-FRP'
pd.concat([pg_frp, pg_plm], ignore_index=True)[
    ['model','gene','label','metric','n','accuracy','mean_prob']
].to_csv(OUT / 'baselines_external_pergene.csv', index=False)

print('\nFRP-XGBoost + PLM-FRP (identical sequences)')
print(res.to_string(index=False))
print('\nContext — external AUROC on the SAME cohort:  ETAP-CLF = 0.8012')
print(f'\nSaved -> {OUT}')
for f in ['baselines_metrics.csv','baselines_random_predictions.csv',
          'baselines_external_predictions.csv','baselines_external_pergene.csv',
          'frp_ifs_curve.csv','plmfrp_ifs_curve.csv']:
    print('  ', f)